# Vehicle counting and short-term peak flow: run the pipeline on Colab

**Before you start:** choose **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.
Edit only the settings cell (step 3).

There are two ways to use this notebook:
- **One clip (steps 5 to 8):** check the counting line, count one video, see its flow chart, and compare with ground truth.
- **All clips (steps 9 and 10):** run every clip listed in `configs/clips.csv`, then evaluate them all.

If Colab disconnects during a long run, reconnect and run the same cells again. Finished chunks are skipped.

### 1. Check the GPU

In [ ]:
!nvidia-smi || echo "No GPU found. Use Runtime > Change runtime type > T4 GPU"


### 2. Connect Google Drive (videos and results live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 3. Settings (edit this cell)
`LINE` is `x1,y1,x2,y2` as fractions of the frame width and height. `0,0.5,1,0.5` is a horizontal line across the middle.
`BIN_SECONDS` is the size of the time bins for the flow chart: 60 for clips of several minutes, 5 to 10 for 1-minute clips.
Set `LIMIT_SECONDS = 0` to process whole videos.

In [ ]:
REPO_URL = "https://github.com/vireakbothkim/peak-hour-traffic.git"   # your team's repo

DATA_DIR   = "/content/drive/MyDrive/traffic"      # shared Drive folder
VIDEOS_DIR = f"{DATA_DIR}/videos"                  # contains ua_detrac/ and ai_city/ subfolders
OUT_DIR    = f"{DATA_DIR}/outputs"
GT_DIR     = f"{DATA_DIR}/ground_truth"

# --- for the single-clip steps (5 to 8) ---
VIDEO = f"{VIDEOS_DIR}/ua_detrac/MVI_41063.mp4"    # one .mp4 file
XML   = f"{DATA_DIR}/annotations/MVI_41063.xml"    # UA-DETRAC annotation file for that clip (ground truth + ignored regions)
LINE  = "0,0.5,1,0.5"                              # counting line, see the preview below

# --- for everything ---
BIN_SECONDS   = 10      # 60 for longer clips
LIMIT_SECONDS = 0       # our clips are 27 to 93 s long, so process them whole. 0 = whole video


### 4. Get the code and install the libraries

In [ ]:
# first run clones the repo, later runs pull the latest version
!test -d /content/repo && git -C /content/repo pull || git clone {REPO_URL} /content/repo
%cd /content/repo
!pip install -q -r requirements.txt


### 5. (One clip) Check the counting line
The red line is where vehicles are counted. **A** and **B** are the two sides: `A_to_B` means a vehicle moved from A to B.
Change `LINE` in step 3 until the line crosses the road cleanly, then run this cell again. Write the final line into `configs/clips.csv`.

In [ ]:
!python src/preview_line.py --video "{VIDEO}" --line "{LINE}" --time-s 5 --out /content/preview.png
from IPython.display import Image, display
display(Image("/content/preview.png"))


### 6. (One clip) Detect, track and count
This is the slow step. Results are saved to Drive chunk by chunk, so it can resume. The frame skipping is chosen automatically from the video's frame rate.

In [ ]:
!python src/detect_track.py --video "{VIDEO}" --out-dir "{OUT_DIR}" --line "{LINE}" --limit-seconds {LIMIT_SECONDS} --ignore-xml "{XML}"


### 7. (One clip) Flow per time bin, peak flow and class mix

In [ ]:
import os
NAME = os.path.splitext(os.path.basename(VIDEO))[0]
CROSSINGS = f"{OUT_DIR}/{NAME}_crossings.csv"
BINS = f"{OUT_DIR}/{NAME}_bins.csv"
CHART = f"{OUT_DIR}/{NAME}_chart.png"

!python src/aggregate.py --crossings "{CROSSINGS}" --bin-seconds {BIN_SECONDS} --out "{BINS}"
!python src/plots.py --bins "{BINS}" --out "{CHART}" --title "{NAME}: vehicle flow"
display(Image(CHART))


### 8. (One clip, UA-DETRAC) Ground truth and error
This builds the true crossings from the annotation file, using the **same line**, then compares them with the system's counts.
It needs a UA-DETRAC annotation XML. If your clip is from AI City, skip this step. Its ground truth has a different format.
Also add a row for this clip in `configs/clips.csv` (even a minimal one) so `evaluate.py` knows about it.

In [ ]:
GT_CSV = f"{GT_DIR}/{NAME}_gt_crossings.csv"
!python src/gt_from_detrac.py --xml "{XML}" --line "{LINE}" --out "{GT_CSV}"

import pandas as pd
system = pd.read_csv(CROSSINGS)
truth = pd.read_csv(GT_CSV)
print(f"System counted {len(system)} vehicles, ground truth has {len(truth)}.")
print("Error: {:.1f}%".format(100 * abs(len(system) - len(truth)) / max(1, len(truth))))


### 9. (All clips) Run every clip in `configs/clips.csv`
Fill in `configs/clips.csv` first (one row per clip, with its counting line and conditions) and push it to GitHub, then run this cell.
A clip that fails is listed at the end, and the others carry on. A summary table of all clips is saved as `all_clips_summary.csv`.

In [ ]:
!python src/run_batch.py --clips configs/clips.csv --videos-dir "{VIDEOS_DIR}" --out-dir "{OUT_DIR}" --bin-seconds {BIN_SECONDS} --limit-seconds {LIMIT_SECONDS} --annotations-dir "{DATA_DIR}/annotations"


### 9b. (All clips, UA-DETRAC) Build the ground truth for every clip
Uses each clip's own counting line from `configs/clips.csv` (empty = default), so the true counts and the system counts use the same line. Run this before step 10.

In [ ]:
import os, subprocess
import pandas as pd

clips = pd.read_csv("configs/clips.csv", comment="#", skip_blank_lines=True)
os.makedirs(GT_DIR, exist_ok=True)
for _, r in clips.iterrows():
    line = r["line"] if isinstance(r["line"], str) and r["line"].strip() else "0,0.5,1,0.5"
    xml = f"{DATA_DIR}/annotations/{r['clip']}.xml"
    if not os.path.exists(xml):
        print("no annotation XML for", r["clip"])
        continue
    result = subprocess.run(["python", "src/gt_from_detrac.py", "--xml", xml, "--line", line,
                             "--out", f"{GT_DIR}/{r['clip']}_gt_crossings.csv"], capture_output=True, text=True)
    print(r["clip"], "->", (result.stdout or result.stderr).strip())


### 10. (All clips) Evaluate against ground truth
Needs `<clip>_gt_crossings.csv` files in the ground-truth folder. Clips without one are skipped.
Writes three tables to the outputs folder: errors by clip, errors grouped by lighting, weather and density, and peak-flow agreement.

In [ ]:
!python src/evaluate.py --clips configs/clips.csv --outputs-dir "{OUT_DIR}" --ground-truth-dir "{GT_DIR}" --bin-seconds {BIN_SECONDS}


### 11. (Optional) Annotated demo clip
Boxes, track IDs, the counting line and live counts, for the demo or as a backup recording.

In [ ]:
DEMO = f"{OUT_DIR}/{NAME}_demo.mp4"
!python src/make_demo.py --video "{VIDEO}" --line "{LINE}" --seconds 20 --out "{DEMO}"
print("Saved to Drive:", DEMO)
